In [1]:
# Flipkart Gridlock 2.0 - Traffic Demand Forecasting
# **Approach:** Autoregressive Time-Series Forecasting via Heterogeneous LightGBM Ensemble

# ### Architecture Overview
# Predicting 47 time-steps into the future requires a model that understands momentum. Standard regression fails here because it cannot dynamically update its historical context. 
# This notebook implements a **Recursive Forecasting Loop**:
# 1. Predict the next 15 minutes.
# 2. Append that prediction to the historical context.
# 3. Dynamically recalculate all rolling averages and lag features.
# 4. Predict the next 15 minutes based on the updated context.

# To prevent recursive error compounding, we utilize a **Heterogeneous Super-Ensemble**, training multiple variations of LightGBM and averaging their step-by-step predictions to ensure extreme stability across the 24-hour cycle.

# # Install pygeohash if not present in the environment
# %pip install pygeohash lightgbm -q

import os
import warnings
import random
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import numpy as np
import pandas as pd
import pygeohash as pgh
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from pathlib import Path

warnings.filterwarnings('ignore')

# --- CONFIGURATION ---
ROOT_DIR = Path.cwd().parent
TRAIN_PATH = ROOT_DIR / 'train.csv'
TEST_PATH = ROOT_DIR / 'test.csv'
SUBMISSION_PATH = ROOT_DIR / 'Submission files Gemini' / 'submission_frankenstein.csv'
print(f"Root Directory: {ROOT_DIR}")
print(f"Train Path: {TRAIN_PATH}")
#print(f"Test Path: {TEST_PATH}")
print(f"Submission Path: {SUBMISSION_PATH}")





Root Directory: e:\VS Code\gridlock_traffic_problem
Train Path: e:\VS Code\gridlock_traffic_problem\train.csv
Submission Path: e:\VS Code\gridlock_traffic_problem\Submission files Gemini\submission_frankenstein.csv


In [2]:
# Expanded window to buffer against missing rows
LAG_FEATURES = [1, 2, 3, 4, 8, 12, 24, 95, 96, 97]
ROLLING_WINDOWS = [4, 8, 12, 24]
DYNAMIC_FEATURE_COLUMNS = [
    *[f'demand_lag_{lag}' for lag in LAG_FEATURES],
    *[f'demand_roll_mean_{window}' for window in ROLLING_WINDOWS],
    *[f'demand_roll_std_{window}' for window in ROLLING_WINDOWS],
]

# --- 2. DATA PREPARATION ---
def parse_time(series):
    parts = series.astype(str).str.split(':', n=1, expand=True)
    return parts[0].astype(int) * 60 + parts[1].astype(int)

def decode_geo(val):
    if pd.isna(val): return np.nan, np.nan
    try:
        dec = pgh.decode(str(val))
        return float(dec[0]), float(dec[1])
    except: return np.nan, np.nan

print("1. Loading and Structuring Data...")
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
train_raw['is_test'] = False
test_raw['is_test'] = True
test_raw['demand'] = np.nan

df = pd.concat([train_raw, test_raw], ignore_index=True, sort=False)
df['time_mins'] = parse_time(df['timestamp']).astype(int)
df['time_sin'] = np.sin(2 * np.pi * df['time_mins'] / 1440.0)
df['time_cos'] = np.cos(2 * np.pi * df['time_mins'] / 1440.0)

geos = df['geohash'].astype('string').dropna().unique()
lookup = {v: decode_geo(v) for v in geos}
coords = df['geohash'].astype('string').map(lookup)
df['latitude'] = [p[0] if isinstance(p, tuple) else np.nan for p in coords]
df['longitude'] = [p[1] if isinstance(p, tuple) else np.nan for p in coords]

df = df.sort_values(['geohash', 'day', 'time_mins']).reset_index(drop=True)

df['Temperature'] = df.groupby('geohash')['Temperature'].transform(lambda s: s.ffill().bfill()).fillna(0)
df['Weather'] = df.groupby('geohash')['Weather'].transform(lambda s: s.ffill().bfill()).fillna('Unknown')
df['RoadType'] = df['RoadType'].fillna('Unknown')
df['LargeVehicles'] = df['LargeVehicles'].fillna('Unknown')
df['Landmarks'] = df['Landmarks'].fillna('Unknown')

print("2. Building Dynamic Lag Features...")
grp = df.groupby('geohash', sort=False)
for lag in LAG_FEATURES:
    df[f'demand_lag_{lag}'] = grp['demand'].shift(lag)

lag_grp = df.groupby('geohash', sort=False)['demand'].shift(1).groupby(df['geohash'], sort=False)
for w in ROLLING_WINDOWS:
    df[f'demand_roll_mean_{w}'] = lag_grp.transform(lambda s: s.rolling(w, min_periods=1).mean())
    df[f'demand_roll_std_{w}'] = lag_grp.transform(lambda s: s.rolling(w, min_periods=1).std())

df[DYNAMIC_FEATURE_COLUMNS] = df[DYNAMIC_FEATURE_COLUMNS].fillna(df.groupby('geohash')['demand'].transform('mean'))

cat_cols = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']
for c in cat_cols: df[c] = df[c].astype('category')
cat_levels = {c: df[c].cat.categories for c in cat_cols}

train_df = df[~df['is_test']].copy()
test_df = df[df['is_test']].copy()

f_cols = ['geohash', 'day', 'time_mins', 'time_sin', 'time_cos', 'latitude', 'longitude', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather'] + DYNAMIC_FEATURE_COLUMNS

valid_train_mask = train_df['demand'].notna()
X_full = train_df.loc[valid_train_mask, f_cols].copy()
y_full = train_df.loc[valid_train_mask, 'demand'].astype(float)

X_train, X_val, y_train, y_val = train_test_split(X_full, y_full, test_size=0.1, random_state=42)

# --- 3. RECURSIVE INFERENCE ---
def get_dyn_feats(hist):
    feats = {}
    for lag in LAG_FEATURES:
        feats[f'demand_lag_{lag}'] = hist[-lag] if len(hist) >= lag else np.nan
    for w in ROLLING_WINDOWS:
        vals = hist[-w:]
        if vals:
            feats[f'demand_roll_mean_{w}'] = float(np.mean(vals))
            feats[f'demand_roll_std_{w}'] = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
        else:
            feats[f'demand_roll_mean_{w}'] = np.nan
            feats[f'demand_roll_std_{w}'] = np.nan
    return feats

def recursive_predict(model, hist_df, tgt_df, framework='lgb'):
    stat_cols = [c for c in f_cols if c not in DYNAMIC_FEATURE_COLUMNS]
    preds, p_idx = [], []
    
    hist_map = {str(g): grp.sort_values(['day', 'time_mins'])['demand'].astype(float).tolist()
                for g, grp in hist_df.groupby('geohash', sort=False)}

    for g_val, g_grp in tgt_df.sort_values(['geohash', 'day', 'time_mins']).groupby('geohash', sort=False):
        g_key = str(g_val)
        h_vals = hist_map.get(g_key, [])
        
        for row in g_grp.to_dict('records'):
            f_row = {c: row[c] for c in stat_cols}
            f_row.update(get_dyn_feats(h_vals))
            f_df = pd.DataFrame([f_row], columns=f_cols)
            
            # Framework specific category handling
            if framework in ['lgb', 'xgb']:
                for c in cat_cols: f_df[c] = pd.Categorical(f_df[c], categories=cat_levels[c])
            elif framework == 'cb':
                for c in cat_cols: f_df[c] = f_df[c].astype(str) # CatBoost handles raw strings best in inference
            
            p = float(np.asarray(model.predict(f_df))[0])
            p = max(0.0, p)
            
            preds.append(p)
            p_idx.append(int(row['Index']))
            h_vals.append(p)

    return pd.Series(preds, index=p_idx)

# --- 4. HETEROGENEOUS TRAINING ---
print("\n3. Training Frankenstein Ensemble...")
N_TRIALS = 10
TOP_K = 4 # We will keep top 4 from EACH of the 3 frameworks (12 Elite Models Total)
rng = random.Random(2024)

# A. LightGBM Pipeline
print("\n--- Phase A: LightGBM ---")
lgb_trials = []
for i in range(N_TRIALS):
    params = {
        'learning_rate': rng.choice([0.02, 0.04, 0.06]),
        'num_leaves': rng.choice([127, 255]),
        'subsample': rng.choice([0.8, 1.0]),
        'colsample_bytree': rng.choice([0.8, 1.0]),
    }
    try:
        model = lgb.LGBMRegressor(objective='regression', metric='rmse', verbosity=-1, device_type='gpu', max_bin=255, n_estimators=1200, random_state=i, **params)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], categorical_feature=cat_cols, callbacks=[lgb.early_stopping(30, verbose=False)])
    except:
        model = lgb.LGBMRegressor(objective='regression', metric='rmse', verbosity=-1, device_type='cpu', n_jobs=-1, n_estimators=1200, random_state=i, **params)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], categorical_feature=cat_cols, callbacks=[lgb.early_stopping(30, verbose=False)])
    
    lgb_trials.append({'model': model, 'rmse': model.best_score_['valid_0']['rmse'], 'type': 'lgb'})
    print(f"  LGBM {i+1}/{N_TRIALS} | RMSE: {lgb_trials[-1]['rmse']:.4f}")

# B. XGBoost Pipeline
print("\n--- Phase B: XGBoost ---")
xgb_trials = []
for i in range(N_TRIALS):
    params = {
        'learning_rate': rng.choice([0.02, 0.04, 0.06]),
        'max_depth': rng.choice([6, 8, 10]),
        'subsample': rng.choice([0.8, 1.0]),
        'colsample_bytree': rng.choice([0.8, 1.0]),
    }
    try:
        model = xgb.XGBRegressor(objective='reg:squarederror', eval_metric='rmse', tree_method='hist', device='cuda', enable_categorical=True, n_estimators=1200, random_state=i, **params)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    except:
        model = xgb.XGBRegressor(objective='reg:squarederror', eval_metric='rmse', tree_method='hist', device='cpu', n_jobs=-1, enable_categorical=True, n_estimators=1200, random_state=i, **params)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    
    # XGBoost doesn't store best_score_ the exact same way, we extract it from eval_results
    best_rmse = min(model.evals_result()['validation_0']['rmse'])
    xgb_trials.append({'model': model, 'rmse': best_rmse, 'type': 'xgb'})
    print(f"  XGB {i+1}/{N_TRIALS} | RMSE: {xgb_trials[-1]['rmse']:.4f}")

# C. CatBoost Pipeline
print("\n--- Phase C: CatBoost ---")
# Pre-cast categories for CatBoost compatibility
cb_X_train = X_train.copy()
cb_X_val = X_val.copy()
for c in cat_cols:
    cb_X_train[c] = cb_X_train[c].astype(str)
    cb_X_val[c] = cb_X_val[c].astype(str)

cb_trials = []
for i in range(N_TRIALS):
    params = {
        'learning_rate': rng.choice([0.03, 0.05]),
        'depth': rng.choice([6, 8]),
        'subsample': rng.choice([0.8, 1.0])
    }
    try:
        model = cb.CatBoostRegressor(loss_function='RMSE', task_type='GPU', iterations=1200, random_seed=i, verbose=False, **params)
        model.fit(cb_X_train, y_train, eval_set=[(cb_X_val, y_val)], cat_features=cat_cols, early_stopping_rounds=30, verbose=False)
    except:
        model = cb.CatBoostRegressor(loss_function='RMSE', task_type='CPU', thread_count=-1, iterations=1200, random_seed=i, verbose=False, **params)
        model.fit(cb_X_train, y_train, eval_set=[(cb_X_val, y_val)], cat_features=cat_cols, early_stopping_rounds=30, verbose=False)
        
    cb_trials.append({'model': model, 'rmse': model.get_best_score()['validation']['RMSE'], 'type': 'cb'})
    print(f"  CatBoost {i+1}/{N_TRIALS} | RMSE: {cb_trials[-1]['rmse']:.4f}")

# --- 5. FRANKENSTEIN BLENDING ---
print("\n4. Selecting Elite Models...")
top_lgb = sorted(lgb_trials, key=lambda x: x['rmse'])[:TOP_K]
top_xgb = sorted(xgb_trials, key=lambda x: x['rmse'])[:TOP_K]
top_cb  = sorted(cb_trials, key=lambda x: x['rmse'])[:TOP_K]

elite_models = top_lgb + top_xgb + top_cb
print(f"Generated Ensemble of {len(elite_models)} diverse models ({TOP_K} per framework).")

print("\n5. Generating Final Super-Ensemble Predictions...")
idx_list = test_df['Index'].astype(int).tolist()
ensemble_test = np.zeros(len(idx_list), dtype=float)

for idx, t in enumerate(tqdm(elite_models, desc="Inference Routing")):
    # Route to the correct recursive handler based on framework type
    p_series = recursive_predict(t['model'], train_df.copy(), test_df.copy(), framework=t['type'])
    
    mapping = {int(k): float(v) for k, v in p_series.to_dict().items()}
    p_arr = np.array([mapping.get(i, 0.0) for i in idx_list], dtype=float)
    ensemble_test += p_arr

ensemble_test /= len(elite_models)

print("\n6. Formatting Final Submission...")
sub = pd.DataFrame({'Index': idx_list, 'demand': ensemble_test}).sort_values('Index').reset_index(drop=True)
sub.to_csv(SUBMISSION_PATH, index=False)
print(f"--- 🏆 FRANKENSTEIN ENSEMBLE READY 🏆 ---")

1. Loading and Structuring Data...
2. Building Dynamic Lag Features...

3. Training Frankenstein Ensemble...

--- Phase A: LightGBM ---
  LGBM 1/10 | RMSE: 0.0266
  LGBM 2/10 | RMSE: 0.0267
  LGBM 3/10 | RMSE: 0.0270
  LGBM 4/10 | RMSE: 0.0268
  LGBM 5/10 | RMSE: 0.0268
  LGBM 6/10 | RMSE: 0.0268
  LGBM 7/10 | RMSE: 0.0270
  LGBM 8/10 | RMSE: 0.0269
  LGBM 9/10 | RMSE: 0.0270
  LGBM 10/10 | RMSE: 0.0266

--- Phase B: XGBoost ---
  XGB 1/10 | RMSE: 0.0271
  XGB 2/10 | RMSE: 0.0281
  XGB 3/10 | RMSE: 0.0277
  XGB 4/10 | RMSE: 0.0277
  XGB 5/10 | RMSE: 0.0273
  XGB 6/10 | RMSE: 0.0282
  XGB 7/10 | RMSE: 0.0293
  XGB 8/10 | RMSE: 0.0281
  XGB 9/10 | RMSE: 0.0287
  XGB 10/10 | RMSE: 0.0273

--- Phase C: CatBoost ---
  CatBoost 1/10 | RMSE: 0.0260
  CatBoost 2/10 | RMSE: 0.0260
  CatBoost 3/10 | RMSE: 0.0260
  CatBoost 4/10 | RMSE: 0.0261
  CatBoost 5/10 | RMSE: 0.0259
  CatBoost 6/10 | RMSE: 0.0257
  CatBoost 7/10 | RMSE: 0.0258
  CatBoost 8/10 | RMSE: 0.0258
  CatBoost 9/10 | RMSE: 0.0262


Inference Routing: 100%|██████████| 12/12 [42:42<00:00, 213.55s/it]


6. Formatting Final Submission...
--- 🏆 FRANKENSTEIN ENSEMBLE READY 🏆 ---


In [2]:
import pandas as pd
from sklearn.metrics import r2_score
from pathlib import Path

ROOT_DIR = Path.cwd().parent
SUBMISSION_PATH = ROOT_DIR / 'Submission files Gemini' / 'submission (2).csv'

actual_df = pd.read_csv(ROOT_DIR / 'submission.csv').sort_values('Index').reset_index(drop=True)
pred_df = pd.read_csv(SUBMISSION_PATH).sort_values('Index').reset_index(drop=True)

r2 = r2_score(actual_df['demand'], pred_df['demand'])
print(f"🚀 MULTI-OUTPUT VECTOR EVALUATION 🚀")
print(f"True Leaderboard Score: {max(0, 100 * r2):.4f} / 100")

🚀 MULTI-OUTPUT VECTOR EVALUATION 🚀
True Leaderboard Score: 96.4616 / 100
